# 03 — 特征筛选链路评审

xc 筛选漏斗:原始 ~1000 维 → Stage-1 统计粗筛 ≤350(`v1_auto`)→ Stage-1.5 null importance 模型精筛 ≤200(`v2_model`,xc 已自动启用)→ 人工去泄漏(`v3_manual_noleak`,runbook 第 3 步)。

本 notebook:各版本清单对比与链路 diff、null importance 结果体检(runbook 第 2.5 步检查点)、入选特征质量回看、5 个 xc 产品间的清单重叠度。非 xc 产品(v1_auto → v2_manual_* 两级)同样可用。

In [ ]:
import sys
sys.path.insert(0, '../src')
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from wdm.config import load_config
from wdm.utils.paths import analysis_dir, report_dir, selected_features_dir

PRODUCT = 'xc_resp_finish'

def read_version(path):
    return [ln.strip() for ln in path.read_text(encoding='utf-8').splitlines()
            if ln.strip() and not ln.startswith('#')]

cfg = load_config(PRODUCT)
d = selected_features_dir(cfg)
versions = {p.stem: read_version(p) for p in sorted(d.glob('*.txt'))}
active = cfg['selected_features']['active_version']
for name, feats in versions.items():
    print('{0}: {1} features{2}'.format(name, len(feats),
                                        '   <- active_version' if name == active else ''))
if active not in versions:
    print('警告:active_version={0} 对应文件不存在(训练会失败)'.format(active))

In [ ]:
# 链路 diff:xc 标准链 v1_auto(统计粗筛)→ v2_model(null importance)→ v3_manual_noleak(人工)
CHAIN = [v for v in ['v1_auto', 'v2_model', 'v3_manual_noleak'] if v in versions]
if len(CHAIN) < 2:
    CHAIN = sorted(versions)   # 非标准链按文件名序兜底
for a, b in zip(CHAIN, CHAIN[1:]):
    sa, sb = set(versions[a]), set(versions[b])
    print('{0} ({1}) -> {2} ({3}): 删 {4} / 增 {5}'.format(
        a, len(sa), b, len(sb), len(sa - sb), len(sb - sa)))
    added = sorted(sb - sa)
    if added:
        print('   新增特征(链路上应为空,出现即检查来源):', added[:20])
if 'v2_model' in versions and 'v3_manual_noleak' in versions:
    removed_manual = sorted(set(versions['v2_model']) - set(versions['v3_manual_noleak']))
    if removed_manual:
        print()
        print('人工剔除({0} 个,应都有泄漏/不可上线理由,备注写在 v3 文件注释里):'.format(
            len(removed_manual)))
        print(removed_manual)

In [ ]:
# Stage-1.5 null importance 体检(runbook 2.5 检查点):
# 被剔除特征的 gain_actual 应明显低于 null_keep_ref;n_written 应接近 final_feature_count
ni_path = report_dir(cfg) / 'null_importance.csv'
meta_path = analysis_dir(cfg) / 'null_importance_meta.json'
if ni_path.is_file():
    ni = pd.read_csv(ni_path)
    if meta_path.is_file():
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        print({k: v for k, v in meta.items() if not isinstance(v, (list, dict))})
        target = cfg['training'].get('final_feature_count')
        n_written = meta.get('n_written')
        if target and n_written is not None and n_written < 0.6 * target:
            print('警告:n_written={0} 远低于 final_feature_count={1} — '
                  '大量候选未过显著性,回查 Stage-1 列表质量'.format(n_written, target))
    kept_m = ni['keep'] == True
    plt.figure(figsize=(6.5, 5))
    plt.scatter(np.log1p(ni.loc[~kept_m, 'null_keep_ref']), np.log1p(ni.loc[~kept_m, 'gain_actual']),
                s=12, alpha=0.5, label='dropped')
    plt.scatter(np.log1p(ni.loc[kept_m, 'null_keep_ref']), np.log1p(ni.loc[kept_m, 'gain_actual']),
                s=12, alpha=0.7, label='kept')
    lim = float(np.log1p(pd.concat([ni['null_keep_ref'], ni['gain_actual']]).max()))
    plt.plot([0, lim], [0, lim], color='gray', lw=0.8, ls='--', label='actual = null ref')
    plt.xlabel('log1p(null_keep_ref)'); plt.ylabel('log1p(gain_actual)')
    plt.title('null importance: actual vs null reference')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    print('keep:', ni['keep'].value_counts().to_dict())
    display(ni.head(10))
else:
    print('无 report/null_importance.csv — 未启用或未跑 Stage-1.5(xc 应由 run_analysis.py 自动跑,'
          '见 runbook 2.5;单独补跑用 scripts/run_model_screen.py)')

In [ ]:
# active 清单 x Stage-1 summary 对表:入选特征的指标分布;
# 清单内但 auto_keep=False(或 summary 无记录)= 人工加回/口径漂移,须有备注理由
sum_path = report_dir(cfg) / 'summary.csv'
if sum_path.is_file() and active in versions:
    summary = pd.read_csv(sum_path)
    sel = pd.DataFrame({'feature': versions[active]}).merge(summary, on='feature', how='left')
    cols = [c for c in ['iv', 'lift_at_k', 'psi', 'missing_rate'] if c in sel.columns]
    print('active={0} 的 {1} 个特征指标分布:'.format(active, len(sel)))
    display(sel[cols].describe().round(4))
    odd = sel[sel['auto_keep'].isna() | (sel['auto_keep'] == False)]
    print('清单内但 Stage-1 非 auto_keep / 无记录: {0} 个'.format(len(odd)))
    if len(odd):
        display(odd[[c for c in ['feature', 'iv', 'lift_at_k', 'psi', 'drop_reason']
                     if c in odd.columns]].head(20))
else:
    print('缺 summary.csv 或 active 清单,跳过')

In [ ]:
# 5 个 xc 产品 active 清单重叠度:响应/资质/端到端各依赖哪些特征。
# 响应与资质清单差异大 → 两段式融合更可能有增益;e2e 与两者的交集可解释其基线表现
XC_PRODUCTS = ['xc_resp_finish', 'xc_qual_finish', 'xc_qual_finish_1v1',
               'xc_e2e_credit', 'xc_e2e_credit_1v1']
lists = {}
for prod in XC_PRODUCTS:
    try:
        c = load_config(prod)
        f = selected_features_dir(c) / (c['selected_features']['active_version'] + '.txt')
        if f.is_file():
            lists[prod] = set(read_version(f))
    except Exception as e:
        print(prod, '跳过:', e)
if len(lists) >= 2:
    names = list(lists)
    mat = pd.DataFrame(index=names, columns=names, dtype=float)
    for a in names:
        for b in names:
            u = lists[a] | lists[b]
            mat.loc[a, b] = len(lists[a] & lists[b]) / len(u) if u else np.nan
    print('Jaccard 重叠矩阵:')
    display(mat.round(3))
    common = set.intersection(*lists.values())
    print('全部 {0} 个产品共有特征: {1} 个'.format(len(lists), len(common)))
    if 'xc_resp_finish' in lists and 'xc_qual_finish' in lists:
        print('仅响应用: {0} | 仅资质 V1 用: {1}'.format(
            len(lists['xc_resp_finish'] - lists['xc_qual_finish']),
            len(lists['xc_qual_finish'] - lists['xc_resp_finish'])))
else:
    print('可对比的 xc 清单不足 2 份(先各自跑完 Stage-1/1.5)')